# Jetsound: Sound detection with Jetson Nano

## Data collection and feature extraction

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
import librosa
import math

DATADIR = "sound_database"
DATASET_PATH = "dataset"
CATEGORIES = ['Claps', 'Keyboard', 'Keys']
snippet_length = 3 #seconds     # X resolution of the spectrograms (basically image width) depends on sampling rate
N_MELS = 128                    # Y resolution of the spectrograms (basically image height)

In [9]:
def convert_to_spectrogram(audio_signal, sample_rate, n_mels, fmax=8000):
    # audio_signal
    # sample_rate
    # n_mels : mels bins (Y-resolution of spectrogram)
    # fmax : maximum frequency in spectrogram

    # Returns a 4D tensorflow tensor (1, n_mels, time_steps, 1)
        
    # Create mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y = audio_signal,
        sr = sample_rate,
        n_mels = n_mels,
        fmax = fmax
    )
    
    # Convert to decibels
    mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

    # Normalize
    mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min())

    # Add channel dimension
    mel_spec = mel_spec[..., np.newaxis]  # shape: (n_mels, time_steps, 1)

    # Add batch dimension
    return mel_spec[np.newaxis, ...]

In [15]:
def split_rec_and_create_dataset():
    # 1. Create the main folder if it doesn't exist
    if not os.path.exists(DATASET_PATH):
        os.makedirs(DATASET_PATH)
        print(f"Created folder: {DATASET_PATH}")
    
    count = 0
    
    for category in CATEGORIES:
        path = os.path.join(DATADIR, category)
        
        if not os.path.exists(path):
            print(f"Skipping {category}: Folder not found at {path}")
            continue

        print(f"Processing category: {category}")

        for fname in os.listdir(path):
            if not fname.lower().endswith('.wav'):
                continue
                
            file_path = os.path.join(path, fname)
            
            # Get the clean filename without extension (e.g. "Keys3")
            base_name = os.path.splitext(fname)[0]
            
            try:
                audio, sr = librosa.load(file_path, sr=None)
                
                total_samples = len(audio)
                samples_per_chunk = int(snippet_length * sr)
                parts = math.ceil(total_samples / samples_per_chunk)
                
                for i in range(parts):
                    start = i * samples_per_chunk
                    end = start + samples_per_chunk
                    
                    chunk_audio = audio[start:end]
                    
                    #Zero Padding
                    if len(chunk_audio) < samples_per_chunk:
                        # Calculate how many zeros we need
                        padding_needed = samples_per_chunk - len(chunk_audio)
                        
                        # Pad the array with zeros (silence) at the end
                        # mode='constant' fills with the constant value (default 0)
                        chunk_audio = np.pad(chunk_audio, (0, padding_needed), mode='constant')
                    
                    # Convert to spectrogram (returns 4D tensor)
                    spectro = convert_to_spectrogram(chunk_audio, sr, N_MELS, fmax=8000)
                    
                    # --- SAVING LOGIC ---
                    
                    # 1. Squeeze the dimensions
                    # Your function returns (1, 128, time, 1), but to save as an image
                    # we need a 2D array: (128, time)
                    image_data = spectro.squeeze()
                    
                    # 2. Construct the filename: "Keys3_8.png"
                    # We use i+1 so the first part is "Keys3_1" instead of "Keys3_0"
                    out_name = f"{base_name}_{i+1}.png"
                    out_path = os.path.join(DATASET_PATH, out_name)
                    
                    # 3. Save as image
                    # origin='lower' puts low frequencies at the bottom (standard for audio)
                    # cmap='viridis' or 'magma' gives it the color. Use 'gray' for black/white.
                    plt.imsave(out_path, image_data, origin='lower', cmap='magma')
                    
                    count += 1
                
            except Exception as e:
                print(f"Error processing {fname}: {e}")
                
    print(f"Done! Saved {count} spectrogram images to '{DATASET_PATH}'")

split_rec_and_create_dataset()

Created folder: dataset
Processing category: Claps
Processing category: Keyboard
Processing category: Keys
Done! Saved 450 spectrogram images to 'dataset'


## Network architecture

## Model training

In [ ]:
# Model definition
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(N_MELS, tensor.shape[2], 1)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(NUMBER_OF_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Validation

You need the library "pyaudio" to test this feature.

    pip install pyaudio
    
    conda install conda-forge::pyaudio

In [ ]:
import pyaudio
import wave

# Configuration parameters
FORMAT = pyaudio.paInt16  # 16-bit resolution
CHANNELS = 1              # Mono recording
RATE = 44100              # Sample rate (44.1kHz is standard)
CHUNK = 1024              # Buffer size
RECORD_SECONDS = 5        # Duration of recording
OUTPUT_FILENAME = "test_recording.wav"

audio = pyaudio.PyAudio()

# Start Recording
stream = audio.open(format=FORMAT, channels=CHANNELS,
                    rate=RATE, input=True,
                    frames_per_buffer=CHUNK)

print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(data)

print("Finished recording.")

# Stop and close the stream
stream.stop_stream()
stream.close()
audio.terminate()

# Save the recorded data as a WAV file 
# (THIS NEEDS TO BE CHANGED SO IT IS STORED IN MEMORY TO BE USED IN REAL TIME)
with wave.open(OUTPUT_FILENAME, 'wb') as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(audio.get_sample_size(FORMAT))
    wf.setframerate(RATE)
    wf.writeframes(b''.join(frames))

print(f"File saved as {OUTPUT_FILENAME}")

Recording...
Finished recording.
File saved as test_recording.wav
